# 02 · Semana 8, segunda parte: del dataset al modelo — Estadísticas de YouTube

**Curso:** Procesamiento de Lenguaje Natural · **Etapa 2 del proyecto: el modelo** · **Entrega:** al finalizar la semana 8

En la primera parte del taller (semana 7) el equipo preparó y analizó los datos. En esta segunda parte, en dos horas, el equipo entrena y compara modelos que predicen el desempeño de un video a partir de sus comentarios y responde, con sus propios resultados, siete preguntas:

1. ¿Qué análisis estadísticos se pueden hacer con el dataset?
2. ¿Cuál fue el modelo seleccionado?
3. ¿Cuál fue su desempeño y bajo qué métricas?
4. ¿Cuántos modelos se probaron?
5. ¿Qué herramientas se usaron para determinar el modelo?
6. ¿Es suficiente un solo modelo?
7. ¿Qué variables se seleccionaron para predecir?

**No hay que escribir código.** Cada sección tiene un formulario a la derecha de la celda (casillas, listas y deslizadores); el equipo toma la decisión, ejecuta la celda e interpreta el resultado. Las cifras y las interpretaciones se registran en la **hoja de resultados** (`docs/Hoja_Resultados_Semana8.docx`).

| Sección | Tiempo | Pregunta |
| --- | --- | --- |
| Celdas 1 y 2 · Preparar el entorno | 5 min | |
| 3 · Cargar la tabla de features | 10 min | |
| 4 · Análisis estadísticos | 20 min | 1 |
| 5 · Selección de variables | 20 min | 7 |
| 6 · Entrenar y comparar modelos | 25 min | 4 y 5 |
| 7 · Desempeño y comparación estadística | 15 min | 3 |
| 8 · Modelo seleccionado | 15 min | 2 |
| 9 · ¿Es suficiente un solo modelo? y 10 · Resumen | 10 min | 6 |

**Antes de ejecutar nada:** *Archivo → Guardar una copia en Drive*. La copia es la que el equipo edita y entrega. Las celdas se ejecutan en orden; si se cambia un formulario, se vuelve a ejecutar esa celda y las siguientes.

### Celda 1 · Descargar el código del taller

**Qué hace:** descarga el código del taller desde GitHub en `/content/youtube-stats-nlp`, instala las librerías que Colab no trae y permite importar el paquete `ytnlp`. No hay que llenar nada.

| Herramienta | Uso en la celda | Parámetro o valor | Qué significa |
| --- | --- | --- | --- |
| `git clone` | Descarga el repositorio | `--depth 1` | Solo la última versión, sin el historial (más rápido) |
| `git clone` | | `-b main` | Rama que se descarga |
| `git fetch` + `git reset --hard` | Actualiza si ya se descargó en la sesión | `FETCH_HEAD` | Deja el código igual a la última versión publicada |
| `subprocess.run` | Ejecuta comandos del sistema desde Python | `capture_output=True`, `text=True` | Guarda la salida del comando como texto para mostrar errores |
| `pip install` | Instala dependencias | `-q -r requirements-colab.txt` | Modo silencioso; instala la lista del archivo |
| `sys.path.insert` | Hace importable el paquete | `0, ".../src"` | Python busca módulos primero en esa carpeta |

**Documentación oficial**

- git clone: https://git-scm.com/docs/git-clone
- subprocess: https://docs.python.org/3/library/subprocess.html
- pip install: https://pip.pypa.io/en/stable/cli/pip_install/
- sys.path: https://docs.python.org/3/library/sys.html#sys.path

In [ ]:
# @title 1 · Descargar el código del taller e instalar dependencias
# Repositorio público del taller (lo define el docente; no se modifica).
REPO_URL = "https://github.com/USUARIO/REPOSITORIO.git"
BRANCH = "main"

import os, subprocess, sys

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git " + args[0] + " falló:\n" + r.stderr[-500:])

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if "USUARIO/REPOSITORIO" in REPO_URL:
        raise ValueError("El repositorio del taller no está configurado; avisar al docente.")
    REPO_DIR = "/content/youtube-stats-nlp"
    if os.path.exists(REPO_DIR):  # actualiza a la última versión publicada
        _git("-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", BRANCH)
        _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
    else:
        _git("clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_DIR}/requirements-colab.txt"], check=True)
    os.chdir(REPO_DIR)
else:  # computador local: el notebook se abre desde la carpeta notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Código del taller listo en", REPO_DIR)

### Celda 2 · Montar Google Drive y cargar los secretos

**Qué hace:** conecta Google Drive (donde se guardan datos, reportes y configuración), lee las credenciales de los Secretos de Colab y muestra un resumen del entorno.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `google.colab.drive.mount` | `mountpoint` | `"/content/drive"` | Carpeta donde aparece Drive dentro de Colab |
| `google.colab.userdata.get` | nombre del secreto | `KAGGLE_API_TOKEN`, `YOUTUBE_API_KEY`, ... | Lee el valor guardado en el panel *Secretos*; se copia a `os.environ` |
| `os.environ` | `YTNLP_STORAGE` | ruta de `CARPETA_DRIVE` | Variable de entorno que indica al código dónde guardar resultados |
| `logging.basicConfig` | `level` | `logging.INFO` | Muestra los mensajes `INFO` y `WARNING` del pipeline |
| | `format` | `"%(levelname)s %(message)s"` | Formato de cada mensaje |
| | `force` | `True` | Reemplaza la configuración de registro que trae Colab |

**Parámetro del formulario:** `CARPETA_DRIVE` es la ruta, dentro de Drive, donde se guarda todo el trabajo del equipo. Si la carpeta compartida tiene otro nombre, se cambia aquí.

**Documentación oficial**

- Archivos y Google Drive en Colab: https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/io.ipynb
- os.environ: https://docs.python.org/3/library/os.html#os.environ
- logging.basicConfig: https://docs.python.org/3/library/logging.html#logging.basicConfig

In [ ]:
# @title 2 · Montar Drive y cargar secretos
CARPETA_DRIVE = "MyDrive/ytnlp-proyecto"  # @param {type:"string"}

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from ytnlp.colab import setup
info = setup(drive_folder=CARPETA_DRIVE)

## 3 · Cargar la tabla de features

**Qué hace:** ejecuta el pipeline de la primera parte con la fuente elegida (ingesta, validación, limpieza, features y análisis estadísticos) y carga la tabla de features: una fila por video, con las variables de contexto, las features de los comentarios, el texto concatenado de los comentarios (`doc`) y las variables objetivo. Si el dataset ya se descargó en la primera parte, se reutiliza desde Drive.

**Parámetros del formulario**

| Parámetro | Opciones | Qué significa |
| --- | --- | --- |
| `SOURCE` | `kaggle`, `api`, `sample` | Fuente de datos. `kaggle` es el dataset completo; `api` usa la última extracción de la YouTube Data API; `sample` es la muestra sintética, solo para practicar |
| `OBJETIVO` | `configuración del equipo`, `engagement`, `log_views`, `log_likes`, `perf_class` | Variable que se va a predecir. Por defecto se usa la que el equipo definió en la primera parte. `engagement` se modela en logaritmo; `perf_class` es una clasificación en tres niveles (bajo, medio, alto) |

| Función | Qué hace |
| --- | --- |
| `ytnlp.pipeline.run(source, skip_model=True)` | Ejecuta las etapas 1 a 5 del pipeline y guarda la tabla en `data/processed/features.parquet` |
| `pd.read_parquet` | Lee la tabla de features desde Drive |

**Documentación oficial:** pandas.read_parquet: https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html

In [ ]:
# @title 3 · Cargar la tabla de features
SOURCE = "kaggle"  # @param ["kaggle", "api", "sample"]
OBJETIVO = "configuración del equipo"  # @param ["configuración del equipo", "engagement", "log_views", "log_likes", "perf_class"]

import json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from ytnlp.config import load_config, path
from ytnlp.pipeline import run
from ytnlp.analysis.stats import COMMENT_FEATS
from ytnlp.models.baseline import CONTEXT_NUM, _target
warnings.filterwarnings("ignore")

run(SOURCE, skip_model=True)
feats = pd.read_parquet(path("processed") / "features.parquet")
TARGET = load_config()["target"] if OBJETIVO == "configuración del equipo" else OBJETIVO
y_all, ES_CLASIF = _target(feats, TARGET)
mask = y_all.notna() & (np.isfinite(y_all) if not ES_CLASIF else True)
df, y = feats[mask].reset_index(drop=True), y_all[mask].reset_index(drop=True)
CTX = [c for c in CONTEXT_NUM if c in df and df[c].notna().any()]
CMT = [c for c in COMMENT_FEATS if c in df]
NOMBRE_Y = {"engagement": "log(engagement)", "log_views": "log(vistas)", "log_likes": "log(likes)",
            "perf_class": "clase de desempeño"}[TARGET]
print(f"Fuente: {SOURCE} · videos con variable objetivo: {len(df):,} · objetivo: {NOMBRE_Y} "
      f"({'clasificación' if ES_CLASIF else 'regresión'})")
display(df[["video_id", "keyword", "doc"] + CMT[:3]].head(3))

## 4 · Análisis estadísticos (pregunta 1)

**Qué hace:** reúne los análisis estadísticos que el pipeline calculó en la primera parte (archivo `reports/stats_report.json`) y agrega dos análisis orientados al modelo:

- **Distribución de la variable objetivo:** forma, asimetría y, si es clasificación, balance de clases.
- **Correlación de Spearman** entre cada feature de los comentarios y la variable objetivo, con la corrección de Benjamini-Hochberg para comparaciones múltiples: al probar muchas variables a la vez, algunas resultan significativas por azar.

| Función | Parámetro | Qué significa |
| --- | --- | --- |
| `scipy.stats.spearmanr` | `nan_policy="omit"` | Correlación por rangos; ignora valores faltantes |
| `scipy.stats.false_discovery_control` | `method="bh"` | Ajusta los valores p por el método de Benjamini-Hochberg |
| `scipy.stats.skew` | | Asimetría: 0 es simétrica; valores altos indican cola larga |

**Documentación oficial**

- scipy.stats.spearmanr: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html
- scipy.stats.false_discovery_control: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html

In [ ]:
# @title 4 · Análisis estadísticos
rep = path("reports") / "stats_report.json"
if rep.exists():
    r = json.loads(rep.read_text(encoding="utf-8"))
    print("Análisis de la primera parte (stats_report):")
    display(pd.DataFrame([{"Análisis": k.replace("_", " "), "Pregunta": v.get("pregunta", ""),
                           "Conclusión": v.get("conclusion", "")} for k, v in r.items()]))

fig, ax = plt.subplots(figsize=(7, 3.2))
if ES_CLASIF:
    y.value_counts().sort_index().plot.bar(ax=ax, color="#4C72B0"); ax.set_title("Videos por clase")
    print("Proporción por clase:", (y.value_counts(normalize=True).round(3) * 100).to_dict())
else:
    ax.hist(y, bins=40, color="#4C72B0"); ax.set_title(f"Distribución de {NOMBRE_Y}")
    print(f"Media {y.mean():.3f} · mediana {y.median():.3f} · desviación {y.std():.3f} · asimetría {stats.skew(y):.2f}")
plt.tight_layout(); plt.show()

yy = y.astype("category").cat.codes if ES_CLASIF else y
filas = []
for c in CMT:
    rho, p = stats.spearmanr(df[c], yy, nan_policy="omit")
    filas.append({"feature": c, "rho": rho, "p": p})
cor = pd.DataFrame(filas)
cor["p_ajustado_BH"] = stats.false_discovery_control(cor["p"].fillna(1), method="bh")
cor["significativa (p ajustado < 0,05)"] = cor["p_ajustado_BH"] < 0.05
print(f"Correlación de Spearman entre features de comentarios y {NOMBRE_Y}"
      + (" (clases ordenadas alfabéticamente)" if ES_CLASIF else "") + ":")
display(cor.sort_values("rho", key=abs, ascending=False).round(4))

## 5 · Selección de variables (pregunta 7)

**Qué hace:** el equipo elige qué bloques de variables usará el modelo y cómo se representa el texto de los comentarios.

| Bloque | Variables | Comentario |
| --- | --- | --- |
| Contexto | `keyword` y variables de fecha y título (`log_age_days`, `pub_dow`, `title_len`, ...) | No usa los comentarios; sirve para medir cuánto aportan |
| Features de comentarios | Sentimiento medio y ponderado, polarización, porcentaje positivo y negativo, longitud, diversidad léxica, preguntas, exclamaciones, emojis | Resumen numérico de los comentarios de cada video |
| Texto TF-IDF | N-gramas del texto concatenado de los comentarios (`doc`) | Representación de PLN: cada palabra o par de palabras es una variable, ponderada por TF-IDF |

**Parámetros del texto**

| Parámetro de `TfidfVectorizer` | Formulario | Qué significa |
| --- | --- | --- |
| `ngram_range` | `NGRAMAS` | `1` usa palabras sueltas; `1-2` agrega pares de palabras consecutivas |
| `max_features` | `MAX_TERMINOS` | Tamaño máximo del vocabulario: se conservan los términos más frecuentes |
| `min_df` | `MIN_DOCUMENTOS` | Un término debe aparecer en al menos ese número de videos para entrar al vocabulario |
| `sublinear_tf` | `True` | Usa 1 + log(frecuencia), para que un término repetido muchas veces no domine |

La celda muestra los términos más asociados con la variable objetivo (prueba F para regresión o chi cuadrado para clasificación).

**Fuga de información:** las features se calculan solo con los comentarios y el contexto del video, nunca con vistas, *likes* ni conteo de comentarios del video, porque con ellos se calcula la variable objetivo.

**Documentación oficial**

- TfidfVectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- Selección de variables (f_regression, chi2): https://scikit-learn.org/stable/modules/feature_selection.html

In [ ]:
# @title 5 · Elegir variables y representación del texto
USAR_CONTEXTO = True  # @param {type:"boolean"}
USAR_FEATURES_COMENTARIOS = True  # @param {type:"boolean"}
USAR_TEXTO_TFIDF = True  # @param {type:"boolean"}
NGRAMAS = "1-2"  # @param ["1", "1-2"]
MAX_TERMINOS = 5000  # @param {type:"slider", min:500, max:20000, step:500}
MIN_DOCUMENTOS = 2  # @param {type:"slider", min:1, max:10, step:1}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import f_regression, chi2
assert USAR_CONTEXTO or USAR_FEATURES_COMENTARIOS or USAR_TEXTO_TFIDF, "Debe elegir al menos un bloque."
NG = (1, 1) if NGRAMAS == "1" else (1, 2)
VARS_NUM = (CTX if USAR_CONTEXTO else []) + (CMT if USAR_FEATURES_COMENTARIOS else [])
BLOQUES = [b for b, u in [("contexto", USAR_CONTEXTO), ("features de comentarios", USAR_FEATURES_COMENTARIOS),
                          ("texto TF-IDF", USAR_TEXTO_TFIDF)] if u]
print("Bloques elegidos:", ", ".join(BLOQUES))
print("Variables numéricas:", len(VARS_NUM), "·", ", ".join(VARS_NUM) if VARS_NUM else "ninguna")
if USAR_TEXTO_TFIDF:
    tf = TfidfVectorizer(ngram_range=NG, max_features=MAX_TERMINOS, min_df=MIN_DOCUMENTOS, sublinear_tf=True)
    Xt = tf.fit_transform(df["doc"].fillna(""))
    voc = np.array(tf.get_feature_names_out())
    print(f"Vocabulario TF-IDF: {len(voc):,} términos")
    if ES_CLASIF:
        score, _ = chi2(Xt, y)
        top = pd.Series(score, index=voc).sort_values(ascending=False).head(15)
        print("Términos más asociados con la clase (chi cuadrado):")
    else:
        score, _ = f_regression(Xt, y)
        corr_sign = np.sign(np.asarray(Xt.multiply(1).T @ (y - y.mean()).values).ravel())
        top = pd.Series(score * corr_sign, index=voc)
        top = pd.concat([top.sort_values().head(8), top.sort_values().tail(8)])
        print("Términos más asociados con valores bajos (negativo) y altos (positivo) del objetivo (prueba F con signo):")
    top.plot.barh(figsize=(7, 4.5), color="#8172B2"); plt.tight_layout(); plt.show()

## 6 · Entrenar y comparar modelos (preguntas 4 y 5)

**Qué hace:** entrena los modelos marcados con los bloques de variables de la sección 5. El 20 % de los videos se reserva para la prueba final y, sobre el 80 % restante, se hace una **validación cruzada de 5 particiones** para comparar los modelos. El split es siempre por video.

| Modelo | Regresión | Clasificación (`perf_class`) | Idea |
| --- | --- | --- | --- |
| Base | `DummyRegressor` (promedio) | `DummyClassifier` (clase más frecuente) | Referencia mínima que cualquier modelo debe superar |
| Lineal regularizado | `Ridge` | `LogisticRegression` | Modelo lineal; funciona bien con TF-IDF, que tiene miles de variables |
| SVM lineal | `LinearSVR` | `LinearSVC` | Máquina de vectores de soporte; también adecuada para texto |
| Bosque aleatorio | `RandomForestRegressor` | `RandomForestClassifier` | Promedio de muchos árboles (*bagging*) |
| Gradient boosting | `HistGradientBoostingRegressor` | `HistGradientBoostingClassifier` | Árboles en secuencia que corrigen errores (*boosting*) |

Para los modelos de árboles, el TF-IDF se reduce a 100 dimensiones (o menos, si el vocabulario es pequeño) con `TruncatedSVD` (análisis semántico latente), porque los árboles no manejan bien miles de columnas dispersas.

| Función | Parámetro | Qué significa |
| --- | --- | --- |
| `train_test_split` | `test_size=0.2`, `random_state=42` | 20 % de los videos para la prueba; la semilla hace reproducible la partición |
| `cross_validate` | `cv=5` | Entrena y evalúa cinco veces sobre el entrenamiento |
| `ColumnTransformer` | | Aplica a cada bloque su preparación (escalado, codificación, TF-IDF) |

**Documentación oficial**

- Validación cruzada: https://scikit-learn.org/stable/modules/cross_validation.html
- ColumnTransformer: https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
- TruncatedSVD: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html

In [ ]:
# @title 6 · Elegir modelos y entrenar
MODELO_BASE = True  # @param {type:"boolean"}
LINEAL_REGULARIZADO = True  # @param {type:"boolean"}
SVM_LINEAL = True  # @param {type:"boolean"}
BOSQUE_ALEATORIO = True  # @param {type:"boolean"}
GRADIENT_BOOSTING = True  # @param {type:"boolean"}

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import LinearSVR, LinearSVC
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.model_selection import train_test_split, cross_validate, KFold, StratifiedKFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score)

def preparacion(arboles):
    partes = []
    if VARS_NUM:
        partes.append(("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), VARS_NUM))
    if USAR_CONTEXTO:
        partes.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=not arboles), ["keyword"]))
    if USAR_TEXTO_TFIDF:
        txt = TfidfVectorizer(ngram_range=NG, max_features=MAX_TERMINOS, min_df=MIN_DOCUMENTOS, sublinear_tf=True)
        partes.append(("txt", make_pipeline(txt, TruncatedSVD(max(2, min(100, len(voc) // 2)), random_state=0)) if arboles else txt, "doc"))
    return ColumnTransformer(partes, sparse_threshold=0 if arboles else 0.3)

if ES_CLASIF:
    catalogo = {"Base": (MODELO_BASE, DummyClassifier(strategy="most_frequent"), False),
                "Lineal regularizado": (LINEAL_REGULARIZADO, LogisticRegression(max_iter=2000), False),
                "SVM lineal": (SVM_LINEAL, LinearSVC(), False),
                "Bosque aleatorio": (BOSQUE_ALEATORIO, RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=0), True),
                "Gradient boosting": (GRADIENT_BOOSTING, HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0), True)}
    METRICA, ESC = "f1_macro", "f1_macro"
else:
    catalogo = {"Base": (MODELO_BASE, DummyRegressor(), False),
                "Lineal regularizado": (LINEAL_REGULARIZADO, Ridge(alpha=1.0), False),
                "SVM lineal": (SVM_LINEAL, LinearSVR(max_iter=5000, random_state=0), False),
                "Bosque aleatorio": (BOSQUE_ALEATORIO, RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=0), True),
                "Gradient boosting": (GRADIENT_BOOSTING, HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=0), True)}
    METRICA, ESC = "MAE", "neg_mean_absolute_error"
MODELOS = {n: Pipeline([("prep", preparacion(a)), ("modelo", m)]) for n, (u, m, a) in catalogo.items() if u}
assert MODELOS, "Debe elegir al menos un modelo."

X = df
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42,
                                          stratify=y if ES_CLASIF else None)
cv = StratifiedKFold(5, shuffle=True, random_state=0) if ES_CLASIF else KFold(5, shuffle=True, random_state=0)

def medir(yt, yp):
    if ES_CLASIF:
        return {"Accuracy": float(accuracy_score(yt, yp)), "F1 macro": float(f1_score(yt, yp, average="macro"))}
    return {"MAE": mean_absolute_error(yt, yp), "RMSE": mean_squared_error(yt, yp) ** 0.5,
            "R²": r2_score(yt, yp), "Spearman": 0.0 if np.allclose(yp, yp[0]) else float(stats.spearmanr(yt, yp)[0])}

filas, ENTRENADOS, PRED, CV_FOLDS = [], {}, {}, {}
for nombre, m in MODELOS.items():
    t0 = time.time()
    r = cross_validate(m, X_tr, y_tr, cv=cv, scoring=ESC)
    folds = r["test_score"] if ES_CLASIF else -r["test_score"]
    CV_FOLDS[nombre] = folds
    m.fit(X_tr, y_tr); p = m.predict(X_te)
    ENTRENADOS[nombre], PRED[nombre] = m, p
    fila = {"Modelo": nombre, f"{METRICA} validación cruzada": folds.mean(), "± desviación": folds.std()}
    fila.update({f"{k} prueba": v for k, v in medir(y_te, p).items()})
    fila["Segundos"] = time.time() - t0
    filas.append(fila)
RESULTADOS = pd.DataFrame(filas).set_index("Modelo").sort_values(f"{METRICA} validación cruzada", ascending=not ES_CLASIF)
print(f"Entrenamiento: {len(y_tr)} videos · prueba: {len(y_te)} videos · modelos probados: {len(MODELOS)}")
print("En validación cruzada, " + ("mayor F1 es mejor." if ES_CLASIF else "menor MAE es mejor."))
display(RESULTADOS.round(4))

## 7 · Desempeño y comparación estadística de modelos (pregunta 3)

**Qué hace:** además de comparar promedios, verifica si las diferencias entre modelos son estadísticamente sólidas.

- **Prueba de Wilcoxon de rangos con signo** sobre las 5 particiones de la validación cruzada: compara el mejor modelo con el segundo y con la base, partición por partición. Con solo cinco particiones la prueba tiene poca potencia, por lo que conviene leer también la diferencia promedio.
- **Intervalo de confianza bootstrap del 95 %** para la métrica del mejor modelo en la prueba: remuestrea los videos de prueba 1.000 veces para estimar cuánto podría variar la métrica con otros datos.

| Métrica | Tipo de problema | Qué significa |
| --- | --- | --- |
| MAE | Regresión | Error absoluto medio en la escala del objetivo (logaritmo); menor es mejor |
| RMSE | Regresión | Penaliza más los errores grandes |
| R² | Regresión | Proporción de la variabilidad explicada; 0 equivale a predecir el promedio |
| Spearman | Regresión | ¿El modelo ordena bien los videos de menor a mayor desempeño? |
| Accuracy | Clasificación | Proporción de videos bien clasificados |
| F1 macro | Clasificación | Promedio del F1 de cada clase; no se deja engañar por clases desbalanceadas |

**Documentación oficial**

- Métricas de scikit-learn: https://scikit-learn.org/stable/modules/model_evaluation.html
- scipy.stats.wilcoxon: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html
- scipy.stats.bootstrap: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html

In [ ]:
# @title 7 · Comparación estadística y bootstrap
orden = list(RESULTADOS.index)
MEJOR = next((n for n in orden if n != "Base"), orden[0])
print(f"Mejor modelo en validación cruzada: {MEJOR}")
def comparar(a, b):
    d = CV_FOLDS[a] - CV_FOLDS[b]
    p = stats.wilcoxon(CV_FOLDS[a], CV_FOLDS[b]).pvalue if np.any(d != 0) else 1.0
    return {"Comparación": f"{a} frente a {b}", f"Diferencia promedio de {METRICA}": d.mean(), "p (Wilcoxon)": p}
comps = [comparar(MEJOR, n) for n in orden if n != MEJOR][:2]
if "Base" in CV_FOLDS and MEJOR != "Base" and not any("Base" in c["Comparación"] for c in comps):
    comps.append(comparar(MEJOR, "Base"))
if comps:
    display(pd.DataFrame(comps).round(4))

yt, yp = np.asarray(y_te), np.asarray(PRED[MEJOR])
def stat(idx):
    idx = np.asarray(idx, dtype=int)
    return f1_score(yt[idx], yp[idx], average="macro") if ES_CLASIF else mean_absolute_error(yt[idx], yp[idx])
boot = stats.bootstrap((np.arange(len(yt)),), stat, n_resamples=1000, vectorized=False, random_state=0)
ic = boot.confidence_interval
print(f"{METRICA} de {MEJOR} en prueba: {stat(np.arange(len(yt))):.4f} · IC 95 % bootstrap: [{ic.low:.4f}, {ic.high:.4f}]")

## 8 · Modelo seleccionado (pregunta 2)

**Qué hace:** el equipo elige el modelo que considera mejor (por error en validación cruzada, estabilidad, tiempo y facilidad de explicarlo). La celda muestra:

- **Términos del texto con más peso** (solo para los modelos lineales con TF-IDF): qué palabras empujan la predicción hacia arriba o hacia abajo.
- **Importancia por permutación por bloque**: cuánto empeora el desempeño al desordenar cada columna original (contexto, features de comentarios y el texto `doc`).
- **Matriz de confusión** (clasificación) o **predicción frente a valor real** (regresión).
- **Error por keyword**, para ver en qué temas falla el modelo.

**Documentación oficial**

- Importancia por permutación: https://scikit-learn.org/stable/modules/permutation_importance.html
- Matriz de confusión: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html

In [ ]:
# @title 8 · Analizar el modelo seleccionado
MODELO_ELEGIDO = "Lineal regularizado"  # @param ["Base", "Lineal regularizado", "SVM lineal", "Bosque aleatorio", "Gradient boosting"]
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay
assert MODELO_ELEGIDO in ENTRENADOS, "Ese modelo no se entrenó en la sección 6: márquelo y vuelva a ejecutarla."
m = ENTRENADOS[MODELO_ELEGIDO]; p = PRED[MODELO_ELEGIDO]
print(f"{MODELO_ELEGIDO}:", {k: round(v, 4) for k, v in medir(y_te, p).items()})

est = m.named_steps["modelo"]; prep = m.named_steps["prep"]
if USAR_TEXTO_TFIDF and hasattr(est, "coef_") and "txt" in prep.named_transformers_:
    nombres = prep.get_feature_names_out()
    coef = np.atleast_2d(est.coef_)[-1] if ES_CLASIF else est.coef_
    s = pd.Series(np.ravel(coef), index=nombres)
    s = s[s.index.str.startswith("txt__")]; s.index = s.index.str.replace("txt__", "")
    print("Términos con más peso" + (f" para la clase '{est.classes_[-1]}'" if ES_CLASIF else "") + ":")
    pd.concat([s.sort_values().head(10), s.sort_values().tail(10)]).plot.barh(figsize=(7, 5), color="#55A868")
    plt.tight_layout(); plt.show()

cols = (["keyword"] + VARS_NUM) if USAR_CONTEXTO else list(VARS_NUM)
cols = list(dict.fromkeys(cols + (["doc"] if USAR_TEXTO_TFIDF else [])))
pi = permutation_importance(m, X_te, y_te, n_repeats=3, random_state=0,
                            scoring="f1_macro" if ES_CLASIF else "neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=X_te.columns)[cols].sort_values()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
imp.tail(12).plot.barh(ax=ax[0], color="#8172B2"); ax[0].set_title("Importancia por permutación")
if ES_CLASIF:
    ConfusionMatrixDisplay.from_predictions(y_te, p, ax=ax[1], colorbar=False); ax[1].set_title("Matriz de confusión")
else:
    ax[1].scatter(y_te, p, s=8, alpha=0.5); lim = [min(y_te.min(), p.min()), max(y_te.max(), p.max())]
    ax[1].plot(lim, lim, "k--", lw=1); ax[1].set_xlabel("Real"); ax[1].set_ylabel("Predicho")
    ax[1].set_title("Predicción frente a valor real")
plt.tight_layout(); plt.show()

e = pd.DataFrame({"keyword": X_te["keyword"].values, "y": np.asarray(y_te), "p": p})
if ES_CLASIF:
    por_kw = e.groupby("keyword").apply(lambda g: pd.Series({"acierto": (g.y == g.p).mean(), "videos": len(g)}))
    col_orden = "acierto"
else:
    por_kw = e.assign(err=(e.y - e.p).abs()).groupby("keyword")["err"].agg(["mean", "size"]).rename(columns={"mean": "MAE", "size": "videos"})
    col_orden = "MAE"
print("Keywords con mejor y peor desempeño del modelo:")
display(pd.concat([por_kw.sort_values(col_orden).head(5), por_kw.sort_values(col_orden).tail(5)]).round(3))

## 9 · ¿Es suficiente un solo modelo? (pregunta 6)

**Qué hace:** compara el mejor modelo individual con un **ensamble** (promedio de las predicciones en regresión o voto por mayoría en clasificación) de los modelos entrenados, sin contar la base, y mide cuánto aporta el texto: entrena el mejor modelo sin el bloque de texto TF-IDF y compara.

Si el ensamble mejora, los modelos se complementan. Si el desempeño varía mucho entre keywords (sección 8), puede convenir un modelo por grupo. Y si la pregunta del negocio cambia (por ejemplo, clasificar el sentimiento de cada comentario en vez de predecir el desempeño del video), se necesita otro modelo.

**Documentación oficial:** ensambles en scikit-learn: https://scikit-learn.org/stable/modules/ensemble.html

In [ ]:
# @title 9 · Ensamble y aporte del texto
cand = [n for n in PRED if n != "Base"]
filas = [{"Modelo": f"Mejor individual ({MEJOR})", **medir(y_te, PRED[MEJOR])}]
if len(cand) >= 2:
    if ES_CLASIF:
        votos = pd.DataFrame({n: PRED[n] for n in cand})
        p_ens = votos.mode(axis=1)[0].values
    else:
        p_ens = np.mean([PRED[n] for n in cand], axis=0)
    filas.append({"Modelo": f"Ensamble de {len(cand)} modelos", **medir(y_te, p_ens)})
else:
    print("Para el ensamble se necesitan al menos dos modelos además de la base (sección 6).")
if USAR_TEXTO_TFIDF and (USAR_CONTEXTO or USAR_FEATURES_COMENTARIOS) and MEJOR != "Base":
    USAR_TEXTO_TFIDF = False
    sin_txt = Pipeline([("prep", preparacion(catalogo[MEJOR][2])), ("modelo", catalogo[MEJOR][1])]).fit(X_tr, y_tr)
    filas.append({"Modelo": f"{MEJOR} sin texto TF-IDF", **medir(y_te, sin_txt.predict(X_te))})
    USAR_TEXTO_TFIDF = True
display(pd.DataFrame(filas).set_index("Modelo").round(4))

## 10 · Resumen para la hoja de resultados

La celda reúne las decisiones y los resultados principales. Se copian en la hoja de resultados y se complementan con la interpretación del equipo.

In [ ]:
# @title 10 · Resumen
r = RESULTADOS.loc[MODELO_ELEGIDO]
resumen = {
    "Fuente de datos": SOURCE,
    "Variable objetivo": f"{NOMBRE_Y} ({'clasificación' if ES_CLASIF else 'regresión'})",
    "Videos (entrenamiento / prueba)": f"{len(y_tr)} / {len(y_te)}",
    "Bloques de variables": ", ".join(BLOQUES),
    "Representación del texto": f"TF-IDF, n-gramas {NGRAMAS}, máximo {MAX_TERMINOS} términos, min_df {MIN_DOCUMENTOS}" if USAR_TEXTO_TFIDF else "no se usó",
    "Modelos probados": f"{len(MODELOS)}: " + ", ".join(MODELOS),
    "Mejor modelo en validación cruzada": MEJOR,
    "Modelo seleccionado por el equipo": MODELO_ELEGIDO,
    "Métricas en prueba del modelo seleccionado": ", ".join(f"{k} {v:.4f}" for k, v in medir(y_te, PRED[MODELO_ELEGIDO]).items()),
    "Herramientas": "Google Colab, Google Drive, Python, pandas, SciPy, scikit-learn (TfidfVectorizer, modelos, validación cruzada, métricas), Matplotlib y el pipeline ytnlp del taller",
}
display(pd.Series(resumen, name="Resultado").to_frame())